# Same-checkpoint timing and executed-operation accounting

Formal DiffJev workflow; see AGENTS.md and prompt.md.

In [ ]:
import os, sys, gc
from pathlib import Path
ROOT = Path(os.environ.get('DIFFJEV_ROOT', Path.cwd())).resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'AGENTS.md').exists(), 'Set DIFFJEV_ROOT to the checked-out repository'
sys.path.insert(0, str(ROOT / 'src'))
from diffjev import workflows as w
from diffjev.config import load_yaml_config
cfg = load_yaml_config(str(ROOT / 'configs/workflow.yaml'))
SEED, DEVICE = cfg['seed'], cfg['device']


Requires one visible A100 80GB, bf16, batch size 1. Each method is loaded and released separately. Every timed request starts from raw semantic text with input caches cleared; five warmups precede 100 fixed stratified queries and three repetitions. Operator/CUDA profiles run separately from wall timing. Inspect unclassified operator coverage before treating logical FLOP totals as complete.

In [ ]:
for backbone in ['nemotron_3b','nemotron_8b','nemotron_14b']:
    w.evaluate_efficiency(backbone,seed=SEED,device=DEVICE)
    gc.collect()
w.evaluate_efficiency('llada_8b_base',seed=SEED,device=DEVICE,attribution=True)

In [ ]:
w.collect_results()

Results include accuracy, survival, p50/p95, query-bootstrap uncertainty, resident-weight peak memory, matched-successful-query speedups, failures, and latency/GFLOP plots. CPU fixtures and theoretical attention ratios are never presented as measured A100 speedups.